## Register organisations and users

Workshop-scale intake: participants fill in `organisation/excel/organisation.xlsx` and `user/excel/user.xlsx`. You review both files and delete any rows you don't want to accept **before** running this notebook. There is no password column in `user.xlsx` - a password is generated automatically and emailed to each accepted user; nobody types or sees it but them.

Run the cells in order, top to bottom.

In [ ]:
# Standard library imports
from os import path, getcwd

import sys 

# Get the working directory of the notebook
notebook_path = getcwd()

# Set the path to the root directory of this jupyter notebook (python) project
module_path = path.abspath(path.join('../..'))

if module_path not in sys.path:
    
    sys.path.append(module_path)

# Package application imports
from src.lib import Initiate_process

# Import the custom class for running processes defined in the ai4sh package
from src.ai4sh import Run_process

# Generates + emails a password for each accepted user, once manage_user.json exists
from src.community import Provision_user_passwords

## Scheme file
Relative to this notebook, in your home directory using tilde (~), or an absolute path.

In [ ]:
scheme_file = '../scheme_ai4sh.json'

## Step 0a - translate territories from excel

In [ ]:
##%%script false --no-raise-error
job_file = 'import_data/utility/job_translate_general_utility.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## Step 0b - insert territories

In [ ]:
#%%script false --no-raise-error
process_file = 'import_data/utility/general/manage_process/manage_territory.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## Step 1 - translate organisations from Excel

Reads `organisation/excel/organisation.xlsx` and writes `organisation/manage_process/manage_organisation.json`, one block per accepted row.

**process**: translate_tabular_data  
**job_file**: job_translate_organisation.json

In [ ]:
job_file = 'user_management/organisation/job_translate_organisation.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:

    Run_process(structured_process_D, scheme_params_D)

## Step 2 - insert organisations

Runs the file generated in step 1 through the registered `manage_organisation` process, which validates and inserts each row into `community.organisation`.

**process**: manage_organisation  
**process_file**: organisation/manage_process/manage_organisation.json

In [ ]:
process_file = 'user_management/organisation/manage_process/manage_organisation.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:

    Run_process(structured_process_D, scheme_params_D)

## Step 3 - translate users from Excel

Reads `user/excel/user.xlsx` and writes `user/manage_process/manage_user.json`. No password column - the next step fills that in.

**process**: translate_tabular_data  
**job_file**: job_translate_user.json

In [ ]:
job_file = 'user_management/user/job_translate_user.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:

    Run_process(structured_process_D, scheme_params_D)

## Step 4 - generate and email passwords

For each user in the file generated in step 3: generates a random password, stores its hash in the file (so step 5 can insert it like any other field), and emails the plaintext password to that user. Rows where the email fails to send are still written with a password hash - check the printed summary and follow up manually for any `emailed: False` row.

In [ ]:
manage_user_json_path = 'user/manage_process/manage_user.json'

summary_L = Provision_user_passwords(path.join(notebook_path, manage_user_json_path))

for row_D in summary_L:

    print(row_D)

## Step 5 - insert users

Runs the password-populated file from step 4 through the registered `manage_user` process, which validates and inserts each row into `community.user`.

**process**: manage_user  
**process_file**: user/manage_process/manage_user.json

In [ ]:
process_file = 'user_management/user/manage_process/manage_user.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:

    Run_process(structured_process_D, scheme_params_D)

## Step 6 - update primary user(s) territory codes

NOT YET IMPLEMENTED